In [ ]:
# uncomment these when running on google colab:
# !pip install torch transformers datasets nltk rouge-score matplotlib
# !git clone <your-repo-url>
# %cd cs4782-lora-replication/code

import sys
sys.path.insert(0, ".")

In [ ]:
import torch
from transformers import GPT2LMHeadModel
from datasets import load_dataset

from lora import inject_lora, merge_lora, LoRAQVWrapper
from data import get_tokenizer, get_dataloaders
from train import run_experiment
from utils import count_parameters

import matplotlib.pyplot as plt
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
dataset = load_dataset("e2e_nlg", trust_remote_code=True)
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")
print(f"\nSample MR:\n  {dataset['train'][0]['meaning_representation']}")
print(f"\nSample Reference:\n  {dataset['train'][0]['human_reference']}")

In [ ]:
# hyperparameters from the lora paper for gpt-2 on e2e nlg
# double check these against the appendix before final runs
CONFIG = {
    "batch_size": 8,
    "max_length": 256,
    "num_epochs": 5,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 500,
    "ranks": [2, 4, 8],
}

In [ ]:
train_loader, val_loader, test_loader, tokenizer = get_dataloaders(
    batch_size=CONFIG["batch_size"],
    max_length=CONFIG["max_length"],
)
print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

In [ ]:
# experiment 1: full fine-tuning baseline
model_ft = GPT2LMHeadModel.from_pretrained("gpt2")
model_ft.resize_token_embeddings(len(tokenizer))

results_ft = run_experiment(
    model=model_ft,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    test_dataset_hf=dataset["test"],
    tokenizer=tokenizer,
    device=device,
    num_epochs=CONFIG["num_epochs"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    warmup_steps=CONFIG["warmup_steps"],
    experiment_name="full_finetune",
)

In [ ]:
# experiments 2-4: lora at different ranks
lora_results = {}

for rank in CONFIG["ranks"]:
    print(f"\n{'='*60}")
    print(f"LoRA rank={rank}")
    print(f"{'='*60}")

    model_lora = GPT2LMHeadModel.from_pretrained("gpt2")
    model_lora.resize_token_embeddings(len(tokenizer))
    inject_lora(model_lora, rank=rank, alpha=rank)

    results = run_experiment(
        model=model_lora,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        test_dataset_hf=dataset["test"],
        tokenizer=tokenizer,
        device=device,
        num_epochs=CONFIG["num_epochs"],
        learning_rate=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
        warmup_steps=CONFIG["warmup_steps"],
        experiment_name=f"lora_r{rank}",
    )
    lora_results[rank] = results

In [ ]:
# comparison table
all_results = [("Full FT", results_ft)] + [(f"LoRA r={r}", lora_results[r]) for r in CONFIG["ranks"]]

print(f"\n{'Experiment':<20} {'Trainable':>12} {'% of Total':>12} {'BLEU':>8} {'ROUGE-L':>8}")
print("-" * 64)

for name, res in all_results:
    p = res["params"]
    m = res["test_metrics"]
    print(f"{name:<20} {p['trainable']:>12,} {p['percentage']:>11.4f}% {m['bleu']:>8.4f} {m['rouge_l']:>8.4f}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for name, res in all_results:
    ax.plot(res["training_history"]["train_loss"], label=name)

ax.set_xlabel("Epoch")
ax.set_ylabel("Training Loss")
ax.set_title("Training Loss: Full Fine-Tuning vs LoRA")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs("../results/figures", exist_ok=True)
plt.savefig("../results/figures/training_loss.png", dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = [name for name, _ in all_results]
bleu_scores = [res["test_metrics"]["bleu"] for _, res in all_results]
rouge_scores = [res["test_metrics"]["rouge_l"] for _, res in all_results]

colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336"]

axes[0].bar(names, bleu_scores, color=colors)
axes[0].set_ylabel("BLEU Score")
axes[0].set_title("BLEU Score Comparison")
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(names, rouge_scores, color=colors)
axes[1].set_ylabel("ROUGE-L Score")
axes[1].set_title("ROUGE-L Score Comparison")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("../results/figures/metrics_comparison.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, res in all_results:
    ax.scatter(
        res["params"]["trainable"],
        res["test_metrics"]["bleu"],
        s=100,
        label=name,
    )
    ax.annotate(name, (res["params"]["trainable"], res["test_metrics"]["bleu"]),
                textcoords="offset points", xytext=(10, 5))

ax.set_xlabel("Trainable Parameters")
ax.set_ylabel("BLEU Score")
ax.set_title("Parameter Efficiency: BLEU vs Trainable Parameters")
ax.set_xscale("log")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/param_efficiency.png", dpi=150)
plt.show()